In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import sys
sys.path.append("../../utils/")

from utils import *

import time

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "CIC18__split__v1__pca4_knn__v1"
CARPETA_DATASET = "CIC18__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG KNN =====
N_NEIGHBORS = 5
WEIGHTS = "distance"      # "uniform" o "distance"
METRIC = "minkowski"      # euclidean suele ser minkowski con p=2
P = 2

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 3

# ===== CONFIG REBALANCEO DENTRO DEL CV =====
ESTRATEGIA_DE_REBALANCEO = "NONE"
TARGET_N = 10000
NEARMISS_VERSION = 1
SMOTE_K_NEIGHBORS = 5
ENN_N_NEIGHBORS = 3

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC18__split__v1__pca4_knn__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(1341149, 55)


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,2,0,44751,3,13,6733,6000,1100,0,41677,...,250,3,0,0,0,0,0,0,0,1
1,37274,4,753825,754,1064,6266,18066,1424,184,20085,...,4725,278,72650,56255,70259,43755,32542,11323,36885,3
2,2,0,4380198,1,3,3,1,3,0,3,...,3,1,0,0,0,0,0,0,0,7
3,624,0,9183,1,3,3,1,3,0,3,...,3,1,0,0,0,0,0,0,0,0
4,2,0,66089,3,13,379,1459,225,0,488,...,156,3,0,0,0,0,0,0,0,2


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
1,360000
0,360000
2,159089
3,116159
4,115628
5,111820
6,75238
7,33125
8,7926


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (1341149, 54)
Shape y_train: (1341149,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("knn", KNeighborsClassifier(
        n_neighbors=N_NEIGHBORS,
        weights=WEIGHTS,
        metric=METRIC,
        p=P,
        n_jobs=-1
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",3
,"copy copy: bool, default=TrueIf False, data passed to fit are 

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
labels_globales = np.array(sorted(y_train.unique()))

resultados_folds = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):

    print("=" * 80)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 80)

    # =========================
    # Split del fold
    # =========================
    df_train_fold = df_train.iloc[train_idx].copy()
    df_val_fold = df_train.iloc[val_idx].copy()

    print("Shape train fold original:", df_train_fold.shape)
    print("Shape val fold original  :", df_val_fold.shape)
    print()

    # =========================
    # Rebalanceo SOLO sobre train fold
    # =========================
    df_train_fold_balanceado = rebalancear_train_fold(
        df_fold_train=df_train_fold,
        label_col=LABEL_COL,
        target_n=TARGET_N,
        random_state=RANDOM_STATE + fold,
        nearmiss_version=NEARMISS_VERSION,
        smote_k_neighbors=SMOTE_K_NEIGHBORS,
        estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
        enn_n_neighbors=ENN_N_NEIGHBORS
    )

    X_train_fold_bal = df_train_fold_balanceado.drop(columns=[LABEL_COL])
    y_train_fold_bal = df_train_fold_balanceado[LABEL_COL]

    X_val_fold = df_val_fold.drop(columns=[LABEL_COL])
    y_val_fold = df_val_fold[LABEL_COL]

    # =========================
    # Modelo nuevo para cada fold
    # =========================
    pipeline_fold = Pipeline([
        ("scaler", RobustScaler()),
        ("pca", PCA(n_components=N_COMPONENTS_PCA)),
        ("knn", KNeighborsClassifier(
            n_neighbors=N_NEIGHBORS,
            weights=WEIGHTS,
            metric=METRIC,
            p=P,
            n_jobs=-1
        ))
    ])

    # =========================
    # Entrenamiento
    # =========================
    t0 = time.time()
    pipeline_fold.fit(X_train_fold_bal, y_train_fold_bal)
    fit_time = time.time() - t0

    # =========================
    # Validación
    # =========================
    t0 = time.time()
    y_pred_val = pipeline_fold.predict(X_val_fold)
    score_time = time.time() - t0

    roc_auc_val = calcular_roc_auc_multiclase_seguro(
        modelo=pipeline_fold,
        X_val=X_val_fold,
        y_val=y_val_fold,
        labels_globales=labels_globales
    )

    metricas_fold = {
        "fold": fold,

        "train_original_rows": int(df_train_fold.shape[0]),
        "train_balanceado_rows": int(df_train_fold_balanceado.shape[0]),
        "val_rows": int(df_val_fold.shape[0]),

        "accuracy": accuracy_score(y_val_fold, y_pred_val),

        "precision_weighted": precision_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),

        "precision_macro": precision_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),

        "mcc": matthews_corrcoef(y_val_fold, y_pred_val),
        "roc_auc": roc_auc_val,

        "fit_time": fit_time,
        "score_time": score_time
    }

    resultados_folds.append(metricas_fold)

    print("Métricas fold:")
    print(metricas_fold)
    print()

FOLD 1/5


Shape train fold original: (1072919, 55)
Shape val fold original  : (268230, 55)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     288000
1     288000
2     127271
3      92927
4      92502
5      89456
6      60191
7      26500
8       6341
9       1107
10       355
11       146
12        54
13        35
14        34
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     288000
1     288000
2     127271
3      92927
4      92502
5      89456
6      60191
7      26500
8       6341
9       1107
10       355
11       146
12        54
13        35
14        34
Name: count, dtype: int64
Shape final: (1072919, 55)



No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined


Métricas fold:
{'fold': 1, 'train_original_rows': 1072919, 'train_balanceado_rows': 1072919, 'val_rows': 268230, 'accuracy': 0.9821720165529583, 'precision_weighted': 0.982546520301122, 'recall_weighted': 0.9821720165529583, 'f1_weighted': 0.9822471970033352, 'precision_macro': 0.9128055342979546, 'recall_macro': 0.9163772926220675, 'f1_macro': 0.9139065055142328, 'mcc': 0.9782331282314816, 'roc_auc': nan, 'fit_time': 3.802417039871216, 'score_time': 0.750197172164917}

FOLD 2/5


Shape train fold original: (1072919, 55)
Shape val fold original  : (268230, 55)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     288000
1     288000
2     127271
3      92927
4      92503
5      89456
6      60190
7      26500
8       6341
9       1107
10       355
11       146
12        54
13        35
14        34
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     288000
1     288000
2     127271
3      92927
4      92503
5      89456
6      60190
7      26500
8       6341
9       1107
10       355
11       146
12        54
13        35
14        34
Name: count, dtype: int64
Shape final: (1072919, 55)



...

Métricas fold:
{'fold': 3, 'train_original_rows': 1072919, 'train_balanceado_rows': 1072919, 'val_rows': 268230, 'accuracy': 0.9821049099653283, 'precision_weighted': 0.9824789848839288, 'recall_weighted': 0.9821049099653283, 'f1_weighted': 0.9821785843251639, 'precision_macro': 0.9226299936827924, 'recall_macro': 0.9214846745426637, 'f1_macro': 0.9184014523043836, 'mcc': 0.9781505229718225, 'roc_auc': nan, 'fit_time': 3.960080146789551, 'score_time': 0.7294037342071533}

FOLD 4/5


Shape train fold original: (1072919, 55)
Shape val fold original  : (268230, 55)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     288000
1     288000
2     127272
3      92927
4      92502
5      89456
6      60190
7      26500
8       6341
9       1108
10       355
11       145
12        53
13        35
14        35
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     288000
1     288000
2     127272
3      92927
4      92502
5      89456
6      60190
7      26500
8       6341
9       1108
10       355
11       145
12        53
13        35
14        35
Name: count, dtype: int64
Shape final: (1072919, 55)



No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 4, 'train_original_rows': 1072919, 'train_balanceado_rows': 1072919, 'val_rows': 268230, 'accuracy': 0.9823658800283339, 'precision_weighted': 0.98281570361148, 'recall_weighted': 0.9823658800283339, 'f1_weighted': 0.9824414840819202, 'precision_macro': 0.9342683786834743, 'recall_macro': 0.9174658972346011, 'f1_macro': 0.9153689577391095, 'mcc': 0.9784865394535328, 'roc_auc': nan, 'fit_time': 2.8162295818328857, 'score_time': 0.8112437725067139}

FOLD 5/5


Shape train fold original: (1072920, 55)
Shape val fold original  : (268229, 55)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     288000
1     288000
2     127271
3      92928
4      92502
5      89456
6      60191
7      26500
8       6340
9       1107
10       356
11       145
12        53
13        36
14        35
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     288000
1     288000
2     127271
3      92928
4      92502
5      89456
6      60191
7      26500
8       6340
9       1107
10       356
11       145
12        53
13        36
14        35
Name: count, dtype: int64
Shape final: (1072920, 55)



No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 5, 'train_original_rows': 1072920, 'train_balanceado_rows': 1072920, 'val_rows': 268229, 'accuracy': 0.9822651540288335, 'precision_weighted': 0.982676486760754, 'recall_weighted': 0.9822651540288335, 'f1_weighted': 0.9823453771014367, 'precision_macro': 0.9098460740778466, 'recall_macro': 0.9120668865908392, 'f1_macro': 0.9107335729294391, 'mcc': 0.978354622994542, 'roc_auc': nan, 'fit_time': 3.280172824859619, 'score_time': 1.0184974670410156}



In [11]:
df_folds = pd.DataFrame(resultados_folds)

df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,1072919,1072919,268230,0.982172,0.982547,0.982172,0.982247,0.912806,0.916377,0.913907,0.978233,NaN,3.802417,0.750197
1,2,1072919,1072919,268230,0.982604,0.982984,0.982604,0.982662,0.968941,0.915115,0.915366,0.978763,NaN,3.039187,0.707180
2,3,1072919,1072919,268230,0.982105,0.982479,0.982105,0.982179,0.922630,0.921485,0.918401,0.978151,NaN,3.960080,0.729404
3,4,1072919,1072919,268230,0.982366,0.982816,0.982366,0.982441,0.934268,0.917466,0.915369,0.978487,NaN,2.816230,0.811244
4,5,1072920,1072920,268229,0.982265,0.982676,0.982265,0.982345,0.909846,0.912067,0.910734,0.978355,NaN,3.280173,1.018497


In [12]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'CIC18__split__v1__pca4_knn__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__train.csv',
 'shape_train': {'rows': 1341149, 'cols': 55},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 3},
 'metricas_media': {'accuracy': 0.98230248836085,
  'precision_weighted': 0.9827003350990994,
  'recall_weighted': 0.98230248836085,
  'f1_weighted': 0.9823748829127601,
  'precision_macro': 0.9296981746306908,
  'recall_macro': 0.9165020226859258,
  'f1_macro': 0.914755349282945,
  'mcc': 0.9783974973987402,
  'roc_auc': nan,
  'fit_time': 3.3796173572540282,
  'score_time': 0.8033044338226318},
 'metricas_std': {'accuracy': 0.00019536642698742807,
  'precision_weighted': 0.0002043207903512721,
  'recall_weighted': 0.00019536642698742807,
  'f1_weighted': 0.00018

In [13]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}\t"
    f"{summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.982302 ± 0.000195	0.982700 ± 0.000204	0.982302 ± 0.000195	0.982375 ± 0.000189	0.929698 ± 0.023929	0.916502 ± 0.003441	0.914755 ± 0.002781	0.978397 ± 0.000240	nan ± nan


In [14]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC18__split__v1__pca4_knn__v1/CIC18__split__v1__pca4_knn__v1__folds.csv


In [15]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC18__split__v1__pca4_knn__v1/CIC18__split__v1__pca4_knn__v1__summary_cv.json


In [16]:
df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,1072919,1072919,268230,0.982172,0.982547,0.982172,0.982247,0.912806,0.916377,0.913907,0.978233,NaN,3.802417,0.750197
1,2,1072919,1072919,268230,0.982604,0.982984,0.982604,0.982662,0.968941,0.915115,0.915366,0.978763,NaN,3.039187,0.707180
2,3,1072919,1072919,268230,0.982105,0.982479,0.982105,0.982179,0.922630,0.921485,0.918401,0.978151,NaN,3.960080,0.729404
3,4,1072919,1072919,268230,0.982366,0.982816,0.982366,0.982441,0.934268,0.917466,0.915369,0.978487,NaN,2.816230,0.811244
4,5,1072920,1072920,268229,0.982265,0.982676,0.982265,0.982345,0.909846,0.912067,0.910734,0.978355,NaN,3.280173,1.018497


In [17]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(335288, 55)


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,4790,0,1343807,1,3,163,1,6,10,180,...,3,3,0,0,0,0,0,0,0,0
1,2,0,13905,3,13,371,1459,261,0,443,...,156,3,0,0,0,0,0,0,0,2
2,2,0,5725,3,13,2844,1459,137,0,4649,...,156,3,0,0,0,0,0,0,0,2
3,2,0,217626,3,13,230,1459,203,0,20004,...,156,3,0,0,0,0,0,0,0,2
4,78205,5,456956,756,1066,16372,72605,1778,443,134586,...,4725,278,72650,56255,70259,43755,32542,11323,36885,0


In [18]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,90000
1,90000
2,39772
3,29040
4,28907
5,27955
6,18810
7,8281
8,1982


In [19]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (335288, 54)
Shape y_test: (335288,)


In [20]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [21]:
print("Rebalanceando todo el train original para entrenar el modelo final...")

df_train_balanceado_final = rebalancear_train_fold(
    df_fold_train=df_train,
    label_col=LABEL_COL,
    target_n=TARGET_N,
    random_state=RANDOM_STATE,
    nearmiss_version=NEARMISS_VERSION,
    smote_k_neighbors=SMOTE_K_NEIGHBORS,
    estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
    enn_n_neighbors=ENN_N_NEIGHBORS
)

X_train_balanceado_final = df_train_balanceado_final.drop(columns=[LABEL_COL])
y_train_balanceado_final = df_train_balanceado_final[LABEL_COL]

pipeline.fit(X_train_balanceado_final, y_train_balanceado_final)

print("Modelo final entrenado con todo el train rebalanceado.")
print("Train original   :", df_train.shape)
print("Train balanceado :", df_train_balanceado_final.shape)

Rebalanceando todo el train original para entrenar el modelo final...


Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     360000
1     360000
2     159089
3     116159
4     115628
5     111820
6      75238
7      33125
8       7926
9       1384
10       444
11       182
12        67
13        44
14        43
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     360000
1     360000
2     159089
3     116159
4     115628
5     111820
6      75238
7      33125
8       7926
9       1384
10       444
11       182
12        67
13        44
14        43
Name: count, dtype: int64
Shape final: (1341149, 55)



Modelo final entrenado con todo el train rebalanceado.
Train original   : (1341149, 55)
Train balanceado : (1341149, 55)


In [22]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

Predicciones en test generadas.
Número de predicciones: 335288


In [23]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.9827103862947675,
 'precision_weighted': 0.9831343342769693,
 'recall_weighted': 0.9827103862947675,
 'f1_weighted': 0.9827887006875785,
 'precision_macro': 0.922057068353286,
 'recall_macro': 0.9184163631056902,
 'f1_macro': 0.9117827305736167,
 'roc_auc': 0.9956866815853146,
 'mcc': 0.9788999166084753}

In [24]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{metricas_test['accuracy']:.6f}\t"
    f"{metricas_test['precision_weighted']:.6f}\t"
    f"{metricas_test['recall_weighted']:.6f}\t"
    f"{metricas_test['f1_weighted']:.6f}\t"
    f"{metricas_test['precision_macro']:.6f}\t"
    f"{metricas_test['recall_macro']:.6f}\t"
    f"{metricas_test['f1_macro']:.6f}\t"
    f"{metricas_test['mcc']:.6f}\t"
    f"{metricas_test['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.982710	0.983134	0.982710	0.982789	0.922057	0.918416	0.911783	0.978900	0.995687


In [25]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,85915,167,413,33,45,2947,30,418,15,2,10,3,2,0,0
1,42,89957,0,0,0,0,0,1,0,0,0,0,0,0,0
2,85,0,39687,0,0,0,0,0,0,0,0,0,0,0,0
3,19,0,0,28999,0,22,0,0,0,0,0,0,0,0,0
4,44,0,0,0,28860,0,0,3,0,0,0,0,0,0,0
5,1255,0,0,15,0,26684,0,1,0,0,0,0,0,0,0
6,3,0,0,0,0,0,18799,3,0,0,0,0,0,0,5
7,126,0,0,0,0,0,0,8138,17,0,0,0,0,0,0
8,14,0,0,0,0,0,0,20,1948,0,0,0,0,0,0
9,0,5,0,0,0,0,0,0,0,341,0,0,0,0,0


In [26]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========
              precision    recall  f1-score   support

           0       0.98      0.95      0.97     90000
           1       1.00      1.00      1.00     90000
           2       0.99      1.00      0.99     39772
           3       1.00      1.00      1.00     29040
           4       1.00      1.00      1.00     28907
           5       0.90      0.95      0.93     27955
           6       1.00      1.00      1.00     18810
           7       0.95      0.98      0.97      8281
           8       0.98      0.98      0.98      1982
           9       0.99      0.99      0.99       346
          10       0.87      0.87      0.87       111
          11       0.86      0.91      0.88        46
          12       0.67      0.24      0.35        17
          13       1.00      1.00      1.00        11
          14       0.64      0.90      0.75        10

    accuracy                           0.98    335288
   macro avg       0.92      0.

In [27]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'CIC18__split__v1__pca4_knn__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__test.csv',
 'shape_test': {'rows': 335288, 'cols': 55},
 'parametros': {'label_col': 'LABEL',
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 3},
 'metricas_test': {'accuracy': 0.9827103862947675,
  'precision_weighted': 0.9831343342769693,
  'recall_weighted': 0.9827103862947675,
  'f1_weighted': 0.9827887006875785,
  'precision_macro': 0.922057068353286,
  'recall_macro': 0.9184163631056902,
  'f1_macro': 0.9117827305736167,
  'mcc': 0.9788999166084753}}

In [28]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC18__split__v1__pca4_knn__v1/CIC18__split__v1__pca4_knn__v1__metricas_test.csv


In [29]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC18__split__v1__pca4_knn__v1/CIC18__split__v1__pca4_knn__v1__confusion_matrix_test.csv


In [30]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/CIC18__split__v1__pca4_knn__v1/CIC18__split__v1__pca4_knn__v1__summary_test.json


In [31]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.98230248836085, 'precision_weighted': 0.9827003350990994, 'recall_weighted': 0.98230248836085, 'f1_weighted': 0.9823748829127601, 'precision_macro': 0.9296981746306908, 'recall_macro': 0.9165020226859258, 'f1_macro': 0.914755349282945, 'mcc': 0.9783974973987402, 'roc_auc': nan, 'fit_time': 3.3796173572540282, 'score_time': 0.8033044338226318}

TEST:
{'accuracy': 0.9827103862947675, 'precision_weighted': 0.9831343342769693, 'recall_weighted': 0.9827103862947675, 'f1_weighted': 0.9827887006875785, 'precision_macro': 0.922057068353286, 'recall_macro': 0.9184163631056902, 'f1_macro': 0.9117827305736167, 'mcc': 0.9788999166084753}
